# 06 — Ten-stage recognition evaluation

This notebook launches the fixed ten-stage recognition evaluator for exported Guesser serving bundles. It scores validation by default; the held-out test split requires a frozen record and must not be used for tuning. This is recognition evaluation, not training, gameplay scoring, or hint-assisted ranking.

The evaluator lives in `models/guesser/recognition/` and runs in the Python 3.13 model project through `uv run`. Do not import its modules into this Python 3.12 notebook kernel.


## Protocol and defaults

Each validation or test original is rendered at ten fixed prefixes: 10%, 20%, …, 100% of its recorded drawing time. Both A/B bundles receive the same preprocessed batch and the same input hash, so the comparison is paired. The end-to-end stage curve counts blank, uncertain and unchanged views as unsuccessful; the policy table treats them as not emitted and reports coverage, accuracy among emitted guesses and overall correctness separately.

By default the runner scores a fixed balanced subset of 50 originals per class: 17,250 originals and 172,500 views per arm. `FULL = True` uses all 172,500 validation originals and 1,725,000 views per arm. It saves full float32 logits so temperature calibration is exact. One scalar temperature per arm is fitted on validation views, and the 0–1 policy thresholds are reported as a trade-off rather than a single game decision.

The A/B comparison uses a stratified bootstrap over whole drawings within class, keeping all ten stages together. The provisional selection goal is B gaining at least +5 mean early macro top-1 points with no more than 2 complete top-1 points lost.


## Launch and development workflow

From the repository root, sync the model environment, then launch JupyterLab through the evaluation environment with the repository as its contents root:

```sh
uv sync --project models/guesser --frozen
uv run --project models/guesser/evaluation jupyter lab --ServerApp.root_dir=.
```

The model scripts are the source of truth. The notebook invokes `recognition.evaluate` as a checked subprocess from `models/guesser/`; it does not copy the runner into the notebook kernel. The subprocess environment removes inherited `PYTHONPATH` and `PYTHONHOME` so unrelated Python installations cannot shadow the model environment.

> **Expected cross-project warning:** `uv` may report that the evaluation `VIRTUAL_ENV` does not match `models/guesser/.venv`. This is intentional: the notebook kernel uses Python 3.12, while the evaluator uses the separate Python 3.13 model environment. `--project` deliberately selects the model environment; do not add `--active`, which would force the notebook environment. This exact mismatch warning is harmless by itself; other errors or nonzero command results still indicate a problem.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from guesser_evaluation import notebook_tools

NOTEBOOK_TOOLS = notebook_tools.module_source_path("notebook_tools.py")
REPOSITORY_ROOT = NOTEBOOK_TOOLS.parents[5]
MODEL_PROJECT = REPOSITORY_ROOT / "models" / "guesser"
if not (MODEL_PROJECT / "recognition" / "evaluate.py").is_file():
    raise FileNotFoundError(f"Ten-stage evaluator not found under {MODEL_PROJECT}")

print("Repository:", REPOSITORY_ROOT)
print("Model project:", MODEL_PROJECT)


## Options

The defaults target the completed production A/B bundles. `SPLIT = "validation"` is the only tuning split. Leave `SAVE_LOGITS = True` for exact temperature calibration and for any later frozen test run. `RESUME = True` is safe for a fresh output root: when `subset.json` is absent, the runner starts a new run; when a valid run exists, it skips class shards whose hashes match `progress.json`. `OVERWRITE = True` replaces the output directory and cannot be combined with resume.

`FREEZE_RECORD` and `VALIDATION_REPORT` are only set for a held-out test run. The freeze record must pin the bundle hashes, temperature and threshold from a completed validation report.

On the current development machine, rendering is the bottleneck at roughly 17 views per second. A 1-per-class smoke run takes about 3 minutes; the default 50-per-class run is roughly 3 hours; `FULL = True` is roughly 28 hours. Inference itself is only a few minutes for the default run.


In [ ]:
from pathlib import Path

# Execution is opt-in; review the options, paths and resource cost.
RUN_EVALUATION = True
DATA_PROFILE = "production"  # 'small', 'medium', or 'production'
SPLIT = "validation"  # 'validation' or 'test'
MODEL_IDS = ("guesser-vit-a-v1", "guesser-vit-b-v1")
LIMIT_PER_CLASS = 50
FULL = False
BATCH_SIZE = 128
BOOTSTRAP = 1000
CALIBRATION_SAMPLES = 50000
SAVE_LOGITS = True
OVERWRITE = False
RESUME = True
DEVICE = "auto"  # 'auto', 'cpu', or 'cuda'
FREEZE_RECORD = None
VALIDATION_REPORT = None

if DATA_PROFILE not in {"small", "medium", "production"}:
    raise ValueError("Select small, medium, or production.")
if SPLIT not in {"validation", "test"}:
    raise ValueError("SPLIT must be validation or test.")
if not MODEL_IDS or len(MODEL_IDS) > 2:
    raise ValueError("MODEL_IDS must contain one or two bundle IDs.")
if LIMIT_PER_CLASS is not None and LIMIT_PER_CLASS < 1:
    raise ValueError("LIMIT_PER_CLASS must be positive or None.")
if BATCH_SIZE < 1 or BOOTSTRAP < 0 or CALIBRATION_SAMPLES < 1:
    raise ValueError("Check BATCH_SIZE, BOOTSTRAP and CALIBRATION_SAMPLES.")
if OVERWRITE and RESUME:
    raise ValueError("OVERWRITE and RESUME are mutually exclusive.")
if SPLIT == "test":
    if FREEZE_RECORD is None or VALIDATION_REPORT is None:
        raise ValueError("A test run requires FREEZE_RECORD and VALIDATION_REPORT.")
    if not SAVE_LOGITS:
        raise ValueError("A test run requires SAVE_LOGITS = True.")
elif FREEZE_RECORD is not None or VALIDATION_REPORT is not None:
    raise ValueError("Freeze inputs are only valid for SPLIT = 'test'.")

PREPARED_DIR = MODEL_PROJECT / "data" / ".local" / "data" / DATA_PROFILE / "prepared"
CHECKPOINT_ROOT = MODEL_PROJECT / "checkpoints" / f"notebook-{DATA_PROFILE}-full"
BUNDLE_ROOT = CHECKPOINT_ROOT / "bundles"
OUTPUT_ROOT = CHECKPOINT_ROOT / "evaluation" / "ten-stage"
if FREEZE_RECORD is not None:
    FREEZE_RECORD = Path(FREEZE_RECORD).expanduser().resolve()
if VALIDATION_REPORT is not None:
    VALIDATION_REPORT = Path(VALIDATION_REPORT).expanduser().resolve()

print("Evaluation enabled:", RUN_EVALUATION)
print("Profile / split:", DATA_PROFILE, SPLIT)
print("Prepared data:", PREPARED_DIR)
print("Bundles:", BUNDLE_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Originals per class:", None if FULL else LIMIT_PER_CLASS)
print("Full validation:", FULL)
print("Batch / bootstrap / calibration samples:", BATCH_SIZE, BOOTSTRAP, CALIBRATION_SAMPLES)
print("Save logits:", SAVE_LOGITS)
print("Overwrite / resume:", OVERWRITE, RESUME)


## Readiness check

This read-only check confirms the selected split manifest and both bundle manifests exist, verifies each manifest's model ID, and probes the model environment with `--no-sync`. It does not load the weights or start evaluation. The evaluator repeats the full bundle hash and label checks before scoring.


In [ ]:
import json
import os
import subprocess

MODEL_ENV = dict(os.environ)
MODEL_ENV.pop("PYTHONPATH", None)
MODEL_ENV.pop("PYTHONHOME", None)
MODEL_ENV["PYTHONUNBUFFERED"] = "1"

manifest_path = PREPARED_DIR / "manifests" / f"{SPLIT}.parquet"
if not manifest_path.is_file():
    raise FileNotFoundError(f"{SPLIT} manifest not found: {manifest_path}")

for model_id in MODEL_IDS:
    bundle_manifest = BUNDLE_ROOT / model_id / "manifest.json"
    if not bundle_manifest.is_file():
        raise FileNotFoundError(f"Bundle manifest not found: {bundle_manifest}")
    payload = json.loads(bundle_manifest.read_text(encoding="utf-8"))
    if payload.get("modelId") != model_id:
        raise ValueError(f"{bundle_manifest} does not describe {model_id}")

probe_code = (
    "import sys, numpy, pyarrow, torch, timm, safetensors, guesser_data; "
    'print("python=" + sys.version.split()[0]); '
    'print("executable=" + sys.executable); '
    'print("torch=" + torch.__version__); '
    'print("cuda_available=" + str(torch.cuda.is_available()))'
)
probe = subprocess.run(
    [
        "uv",
        "run",
        "--project",
        str(MODEL_PROJECT),
        "--frozen",
        "--no-sync",
        "python",
        "-c",
        probe_code,
    ],
    cwd=MODEL_PROJECT,
    env=MODEL_ENV,
    capture_output=True,
    text=True,
    check=False,
)
if probe.returncode != 0:
    raise RuntimeError("Model environment probe failed:\n" + probe.stderr)
print(probe.stdout.strip())
print("Prepared manifest:", manifest_path)
print("Bundles:", ", ".join(MODEL_IDS))
print("Output root:", OUTPUT_ROOT)


## Run the evaluator (explicit opt-in)

When enabled, the cell launches `recognition.evaluate`. The runner validates every bundle hash and label file, selects the fixed subset, streams the ten views through both arms, and writes class shards and logits before computing metrics. Interrupting the cell asks the evaluator to stop; a later `RESUME = True` run skips class shards whose hashes match `progress.json`. If no `subset.json` exists yet, `RESUME = True` starts a new run instead of failing. Do not run two evaluators against the same output root at the same time.


In [ ]:
if not RUN_EVALUATION:
    print("Evaluation disabled. Review the options, then set RUN_EVALUATION = True to launch.")
else:
    evaluation_command = [
        "uv",
        "run",
        "--project",
        str(MODEL_PROJECT),
        "--frozen",
        "--no-sync",
        "python",
        "-m",
        "recognition.evaluate",
        "--prepared-dir",
        str(PREPARED_DIR),
        "--bundle-root",
        str(BUNDLE_ROOT),
        "--output-dir",
        str(OUTPUT_ROOT),
        "--split",
        SPLIT,
        "--batch-size",
        str(BATCH_SIZE),
        "--bootstrap",
        str(BOOTSTRAP),
        "--calibration-samples",
        str(CALIBRATION_SAMPLES),
        "--device",
        DEVICE,
    ]
    for model_id in MODEL_IDS:
        evaluation_command.extend(["--model-id", model_id])
    if FULL:
        evaluation_command.append("--full")
    elif LIMIT_PER_CLASS is not None:
        evaluation_command.extend(["--limit-per-class", str(LIMIT_PER_CLASS)])
    if not SAVE_LOGITS:
        evaluation_command.append("--no-save-logits")
    if OVERWRITE:
        evaluation_command.append("--overwrite")
    if RESUME:
        evaluation_command.append("--resume")
    if SPLIT == "test":
        evaluation_command.extend(
            [
                "--freeze-record",
                str(FREEZE_RECORD),
                "--validation-report",
                str(VALIDATION_REPORT),
            ]
        )
    print("Launching:", " ".join(evaluation_command), flush=True)
    if not notebook_tools.run_interruptible_command(
        evaluation_command, cwd=MODEL_PROJECT, env=MODEL_ENV
    ):
        raise RuntimeError("Evaluation interrupted; rerun with RESUME = True to continue.")
    print("Evaluation command completed successfully.")


## Summary and stage curves

The summary is read from `summary.json`, not from training curves. The stage table shows end-to-end top-1 and top-3 at each prefix. The summary table shows mean early macro/overall top-1, complete top-1/top-3, and blank/uncertain rates. The policy table separates coverage from accuracy among emitted guesses. The A/B section reports paired bootstrap intervals and whether the provisional goal was met.


In [ ]:
import json

import pandas as pd
from IPython.display import display

SUMMARY_PATH = OUTPUT_ROOT / "summary.json"
if not SUMMARY_PATH.is_file():
    print("No summary yet. Run the evaluator or check OUTPUT_ROOT.")
else:
    summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    rows = []
    for model_id, model in summary["models"].items():
        rows.append(
            {
                "model": model_id,
                "mean early top-1 (macro)": model["meanEarlyTop1Macro"],
                "mean early top-1 (overall)": model["meanEarlyTop1"],
                "complete top-1": model["completeTop1"],
                "complete top-3": model["completeTop3"],
                "blank": model["blankRate"],
                "uncertain": model["uncertainRate"],
            }
        )
    display(pd.DataFrame(rows).set_index("model"))
    print("Split:", summary["split"])
    print("Originals / views:", summary["originals"], "/", summary["views"])
    print("Calibration:", summary["calibration"])
    if summary["comparison"]:
        print("A/B comparison:", json.dumps(summary["comparison"], indent=2))


In [ ]:
import matplotlib.pyplot as plt

if SUMMARY_PATH.is_file():
    summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    stages = list(range(1, 11))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for model_id, model in summary["models"].items():
        axes[0].plot(stages, model["stageTop1"], marker="o", label=model_id)
        axes[1].plot(stages, model["stageTop3"], marker="o", label=model_id)
    axes[0].set_title("End-to-end top-1")
    axes[1].set_title("End-to-end top-3")
    for axis in axes:
        axis.set_xlabel("Drawing prefix (%)")
        axis.set_xticks(stages)
        axis.set_xticklabels([str(stage * 10) for stage in stages])
        axis.set_ylim(0, 1)
        axis.grid(True, alpha=0.3)
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[1].legend()
    fig.suptitle("Guesser ten-stage recognition")
    plt.tight_layout()
    plt.show()
else:
    print("No summary yet; run the evaluator first.")


In [ ]:
if SUMMARY_PATH.is_file():
    summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    for model_id, model in summary["models"].items():
        print(f"\n{model_id}")
        policy = pd.DataFrame(model["policy"])
        policy = policy[
            ["threshold", "coverage", "accuracyEmitted", "overallCorrectness"]
        ]
        policy["threshold"] = policy["threshold"].map(lambda value: f"{value:.2f}")
        display(policy)
else:
    print("No summary yet; run the evaluator first.")


## Optional freeze record for a held-out test run

A test run must not fit temperature or choose a threshold. After a validation run, review `report.md` and select one policy threshold per arm. This optional cell writes a freeze record from the validation `summary.json` and `report.md`; it is disabled by default. It records the validation report hash and the bundle identity fields already stored in the summary. Do not edit the record after seeing test results.


In [ ]:
import hashlib
from datetime import UTC, datetime

WRITE_FREEZE_RECORD = False
FROZEN_THRESHOLDS = {
    "guesser-vit-a-v1": 0.5,
    "guesser-vit-b-v1": 0.5,
}
FREEZE_OUTPUT = CHECKPOINT_ROOT / "evaluation" / "ten-stage-freeze.json"

if not WRITE_FREEZE_RECORD:
    print("Freeze record disabled. Set WRITE_FREEZE_RECORD = True after reviewing validation.")
elif not SUMMARY_PATH.is_file() or not (OUTPUT_ROOT / "report.md").is_file():
    raise FileNotFoundError("Run validation first; summary.json and report.md are required.")
else:
    summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    if summary["split"] != "validation":
        raise ValueError("A freeze record must be built from a validation run.")
    if summary["calibration"]["mode"] != "temperature":
        raise ValueError("A freeze record requires a temperature-calibrated validation run.")
    for model_id, threshold in FROZEN_THRESHOLDS.items():
        if not 0 <= threshold <= 1:
            raise ValueError(f"Threshold for {model_id} must be in [0, 1].")
    report_path = OUTPUT_ROOT / "report.md"
    models = {}
    for model_id, model in summary["models"].items():
        if model_id not in FROZEN_THRESHOLDS:
            raise ValueError(f"Set FROZEN_THRESHOLDS for {model_id}.")
        identity = model["identity"]
        models[model_id] = {
            "artifactRevision": identity["artifactRevision"],
            "weightsSha256": identity["weightsSha256"],
            "labelsSha256": identity["labelsSha256"],
            "preprocessing": identity["preprocessing"],
            "preprocessingHash": identity["preprocessingHash"],
            "temperature": summary["calibration"]["temperatures"][model_id],
            "threshold": FROZEN_THRESHOLDS[model_id],
        }
    payload = {
        "schema": "guesser-ten-stage-freeze-v1",
        "createdUtc": datetime.now(UTC).isoformat(),
        "validationReportSha256": hashlib.sha256(report_path.read_bytes()).hexdigest(),
        "models": models,
    }
    FREEZE_OUTPUT.write_text(
        json.dumps(payload, indent=2) + "\n", encoding="utf-8"
    )
    print("Wrote", FREEZE_OUTPUT)


## Artifacts and limits

The output root contains `subset.json`, `progress.json`, `predictions/<model-id>/class-*.parquet`, `logits/<model-id>.npy`, `confusions/<model-id>.npy`, `per-class-<model-id>.csv`, `metrics.json`, `summary.json`, and `report.md`. These are local run artifacts under ignored `checkpoints/`; do not commit them.

- Validation is the tuning split. The held-out test split is a separate frozen run and must not guide model or threshold changes.
- Hint-assisted evaluation is deferred; this runner is sketch-only.
- These are recognition metrics, not game scores. A first human solve does not end a turn, and AI points are a comparison ledger.
- A high pass rate can hide a weak guesser; read coverage, accuracy among emitted guesses and overall correctness together.
- The 345-class classifier is closed-set; low confidence is not a guaranteed unknown-concept detector.


In [ ]:
REPORT_PATH = OUTPUT_ROOT / "report.md"
if REPORT_PATH.is_file():
    print(REPORT_PATH.read_text(encoding="utf-8"))
else:
    print("No report yet; run the evaluator first.")
